# TIL — EDU-ORCH-002: Routing Evidence Matrix

Este experimento mede uma arquitetura de **routing/cascade real** entre:

- **TF-IDF + Multinomial Naive Bayes**;
- **DistilBERT multilingual**.

A pergunta experimental é:

> Quando vale a pena aceitar a decisão do baseline clássico e quando vale a pena escalar a entrada para o Transformer?

O experimento preserva o mesmo dataset, split e configuração de modelos do `EDU-ORCH-001`, mas mede agora uma arquitetura composta.

> **Política TIL:** nenhuma linha será marcada como `measured` se for derivada apenas por interpolação entre métricas anteriores.


## Modo de uso deste notebook

Este notebook é a **origem auditável da evidência**, não o caminho principal do aluno.

```text
AUTHOR / EVIDENCE
→ treinamento
→ medição
→ artefatos versionados
→ consumo pela Aula 13C

STUDENT
→ Aula 13C
→ carrega evidência pronta
→ explora routing, thresholds, latência, custo e utility
```

O tempo de treinamento não é uma meta pedagógica. A latência medida dos sistemas, sim.

📚 Glossário Vivo: [Evidência](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#evidência) · [Reprodutibilidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#reprodutibilidade) · [Modelo pré-treinado](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#modelo-pré-treinado) · [Fine-tuning](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#fine-tuning) · [Roteamento de modelos](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Quality gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Taxa de escalonamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento).


## 1. Contrato experimental

Mantemos:

- dataset: Olist / Polarity;
- folds 1–8: treino;
- fold 9: validação;
- fold 10: teste;
- qualidade principal: `f1_macro`;
- baseline: TF-IDF + MultinomialNB;
- Transformer: DistilBERT multilingual;
- thresholds: `0.60`, `0.70`, `0.80`, `0.90`;
- custo: proxy computacional em segundos por 1.000 exemplos;
- latência: end-to-end, por exemplo, em `batch_size=1`.

Regra de routing:

```python
confidence = max(predict_proba_baseline)

if confidence >= threshold:
    final_prediction = baseline_prediction
else:
    final_prediction = transformer_prediction
```


In [ ]:
from __future__ import annotations

import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
THRESHOLDS = [0.60, 0.70, 0.80, 0.90]

random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
RUN_MODE = "EVIDENCE" if IS_KAGGLE else "SMOKE"

print("RUN_MODE:", RUN_MODE)
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Thresholds:", THRESHOLDS)


## 2. Recursos

No Kaggle, este notebook espera:

- dataset `fredericods/ptbr-sentiment-analysis-datasets`;
- modelo `goddiao/distilbert-base-multilingual-cased`, PyTorch/default/version 1;
- Internet OFF.

A execução local sem esses recursos entra em `SMOKE` e não gera evidência falsa.


In [ ]:
DATASET_ROOT = Path("/kaggle/input")
MODEL_DIR = Path(
    "/kaggle/input/models/goddiao/distilbert-base-multilingual-cased/"
    "pytorch/default/1/distilbert-base-multilingual-cased"
)

def find_olist_csv(root: Path) -> Path | None:
    if not root.exists():
        return None
    candidates = [p for p in root.rglob("*.csv") if "olist" in p.name.lower()]
    if not candidates:
        return None
    candidates = sorted(
        candidates,
        key=lambda p: (p.name.lower() != "olist.csv", len(str(p)))
    )
    return candidates[0]

OLIST_CSV = find_olist_csv(DATASET_ROOT) if IS_KAGGLE else None
RESOURCES_READY = bool(OLIST_CSV and OLIST_CSV.exists() and MODEL_DIR.exists())

print("OLIST_CSV:", OLIST_CSV)
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESOURCES_READY:", RESOURCES_READY)


In [ ]:
EXPECTED_COLUMNS = {
    "review_text", "polarity", "rating", "kfold_polarity"
}

if RESOURCES_READY:
    df = pd.read_csv(OLIST_CSV)
    missing = EXPECTED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Dataset incompatível. Colunas ausentes: {sorted(missing)}")

    work = df.loc[df["polarity"].notna(), [
        "review_text", "polarity", "rating", "kfold_polarity"
    ]].copy()

    work["review_text"] = work["review_text"].astype(str)
    work["polarity"] = work["polarity"].astype(int)
    work["kfold_polarity"] = work["kfold_polarity"].astype(int)

    train_df = work[work["kfold_polarity"].between(1, 8)].reset_index(drop=True)
    val_df = work[work["kfold_polarity"].eq(9)].reset_index(drop=True)
    test_df = work[work["kfold_polarity"].eq(10)].reset_index(drop=True)

    print("train:", len(train_df))
    print("validation:", len(val_df))
    print("test:", len(test_df))

    assert len(test_df) == 3807, f"Test set inesperado: {len(test_df)}"
else:
    df = work = train_df = val_df = test_df = None
    print("SMOKE: recursos Kaggle não disponíveis.")


In [ ]:
if RESOURCES_READY:
    bad_neg = work.loc[work["rating"].isin([1, 2]) & work["polarity"].ne(0)]
    bad_pos = work.loc[work["rating"].isin([4, 5]) & work["polarity"].ne(1)]
    rating3 = work.loc[work["rating"].eq(3)]

    assert bad_neg.empty
    assert bad_pos.empty
    assert rating3.empty
    print("Target integrity gate: PASS")
else:
    print("Target integrity gate: SKIPPED (SMOKE)")


## 3. Baseline e confiança de routing

O baseline é treinado com a mesma configuração do `EDU-ORCH-001`.

Além da classe prevista, agora utilizamos `predict_proba` para obter a confiança que decide o routing.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

baseline_pipeline = None

if RESOURCES_READY:
    baseline_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True,
        )),
        ("nb", MultinomialNB(alpha=1.0)),
    ])

    baseline_pipeline.fit(train_df["review_text"], train_df["polarity"])
    print("Baseline trained.")
else:
    print("Baseline training: SKIPPED (SMOKE)")


## 4. Transformer

O `EDU-ORCH-001` não persiste o classificador fine-tuned como artefato reutilizável; portanto, para manter equivalência experimental, este notebook refaz o treinamento com a mesma configuração.

A execução oficial do `EDU-ORCH-002` permanece em **CPU** para preservar comparabilidade com a evidência já coletada e evitar a incompatibilidade observada com Tesla P100 / `sm_60`.


In [ ]:
transformer_model = None
tokenizer = None
trainer = None

if RESOURCES_READY:
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    set_seed(SEED)

    device = torch.device("cpu")
    print("device:", device)

    tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
    )

    class ReviewDataset(Dataset):
        def __init__(self, frame, tokenizer, max_length=128):
            self.texts = frame["review_text"].tolist()
            self.labels = frame["polarity"].astype(int).tolist()
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            item = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            item = {k: v.squeeze(0) for k, v in item.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    train_ds = ReviewDataset(train_df, tokenizer)
    val_ds = ReviewDataset(val_df, tokenizer)

    id2label = {0: "negative", 1: "positive"}
    label2id = {"negative": 0, "positive": 1}

    set_seed(SEED)
    transformer_model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    args = TrainingArguments(
        output_dir="/kaggle/working/edu-orch-002-distilbert",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to=[],
        seed=SEED,
        data_seed=SEED,
        use_cpu=True,
        fp16=False,
    )

    trainer = Trainer(
        model=transformer_model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
    )

    trainer.train()
    transformer_model.eval()
    print("Transformer trained on CPU.")
else:
    print("Transformer training: SKIPPED (SMOKE)")


In [ ]:
# Persistência do modelo fine-tuned
if RESOURCES_READY:
    from pathlib import Path
    import shutil

    fine_tuned_dir = Path("/kaggle/working/fine-tuned-model")
    fine_tuned_dir.mkdir(parents=True, exist_ok=True)

    transformer_model.save_pretrained(fine_tuned_dir)
    tokenizer.save_pretrained(fine_tuned_dir)

    archive = shutil.make_archive(
        "/kaggle/working/edu-orch-002-distilbert-finetuned",
        "zip",
        root_dir=fine_tuned_dir,
    )

    print("Fine-tuned model saved:", fine_tuned_dir)
    print("Model bundle:", archive)
else:
    print("Fine-tuned model persistence: SKIPPED (SMOKE)")


## 5. Funções de inferência

A medição é separada em duas camadas:

1. **qualidade + runtime proxy**: execução em lote sobre o conjunto de teste;
2. **latência**: execução end-to-end por exemplo (`batch_size=1`) em amostra fixa.

Para cada threshold, o Transformer é executado **somente** nos exemplos escalados.


In [ ]:
def transformer_predict_frame(frame):
    if len(frame) == 0:
        return np.array([], dtype=int)

    ds = ReviewDataset(frame, tokenizer)
    result = trainer.predict(ds)
    return np.argmax(result.predictions, axis=-1).astype(int)


def route_batch(frame, threshold):
    texts = frame["review_text"]
    y_true = frame["polarity"].to_numpy(dtype=int)

    t0 = time.perf_counter()

    baseline_proba = baseline_pipeline.predict_proba(texts)
    baseline_pred = np.argmax(baseline_proba, axis=1).astype(int)
    confidence = np.max(baseline_proba, axis=1)

    escalate_mask = confidence < threshold
    final_pred = baseline_pred.copy()

    if escalate_mask.any():
        escalated_frame = frame.loc[escalate_mask].reset_index(drop=True)
        transformer_pred = transformer_predict_frame(escalated_frame)
        final_pred[escalate_mask] = transformer_pred

    elapsed = time.perf_counter() - t0

    return {
        "y_true": y_true,
        "final_pred": final_pred,
        "escalate_mask": escalate_mask,
        "elapsed_seconds": elapsed,
    }


In [ ]:
LATENCY_SAMPLE_SIZE = 256
WARMUP = 20

def predict_transformer_one(text):
    import torch

    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )

    with torch.no_grad():
        logits = transformer_model(**encoded).logits

    return int(torch.argmax(logits, dim=-1).item())


def route_one(text, threshold):
    proba = baseline_pipeline.predict_proba([text])[0]
    baseline_pred = int(np.argmax(proba))
    confidence = float(np.max(proba))

    if confidence >= threshold:
        return baseline_pred, False

    return predict_transformer_one(text), True


def summarize_ms(values):
    arr = np.asarray(values, dtype=float)
    return {
        "latency_ms": float(arr.mean()),
        "latency_p50_ms": float(np.percentile(arr, 50)),
        "latency_p95_ms": float(np.percentile(arr, 95)),
    }


## 6. Matriz experimental

Cada threshold produz uma linha independente de evidência.

A quality é calculada sobre a decisão final do cascade. A taxa de escalonamento é observada diretamente.


In [ ]:
routing_rows = []

if RESOURCES_READY:
    latency_df = test_df.sample(
        n=min(LATENCY_SAMPLE_SIZE, len(test_df)),
        random_state=SEED,
    ).reset_index(drop=True)

    for threshold in THRESHOLDS:
        print(f"\nThreshold {threshold:.2f}")

        # Batch measurement: quality + runtime proxy
        batch_result = route_batch(test_df, threshold)

        y_true = batch_result["y_true"]
        final_pred = batch_result["final_pred"]
        escalate_mask = batch_result["escalate_mask"]
        elapsed = batch_result["elapsed_seconds"]

        escalation_rate = float(escalate_mask.mean())
        baseline_share = 1.0 - escalation_rate
        transformer_share = escalation_rate

        f1 = float(f1_score(y_true, final_pred, average="macro"))
        acc = float(accuracy_score(y_true, final_pred))
        runtime_per_1000 = float(elapsed / len(test_df) * 1000)

        # Warm-up end-to-end
        for text in latency_df["review_text"].iloc[:min(WARMUP, len(latency_df))]:
            route_one(text, threshold)

        # Single-example end-to-end latency
        times_ms = []
        latency_escalations = 0

        for text in latency_df["review_text"]:
            t0 = time.perf_counter()
            _, escalated = route_one(text, threshold)
            times_ms.append((time.perf_counter() - t0) * 1000)
            latency_escalations += int(escalated)

        lat = summarize_ms(times_ms)

        row = {
            "system": f"cascade_t{int(round(threshold * 100)):03d}",
            "threshold": threshold,
            "quality": f1,
            "quality_metric": "f1_macro",
            "accuracy": acc,
            "escalation_rate": escalation_rate,
            "baseline_share": baseline_share,
            "transformer_share": transformer_share,
            "cost_per_1000": runtime_per_1000,
            "cost_unit": "runtime_seconds_per_1000",
            "cost_method": "measured_batch_runtime_proxy",
            **lat,
            "latency_sample_size": len(latency_df),
            "latency_escalation_rate": latency_escalations / len(latency_df),
        }

        routing_rows.append(row)
        print(row)

    routing_df = pd.DataFrame(routing_rows)
    display(routing_df)
else:
    routing_df = pd.DataFrame()
    print("Routing matrix: SKIPPED (SMOKE)")


## 7. Gates de evidência e exportação

O CSV só é exportado se:

- os quatro thresholds forem executados;
- o test set tiver 3807 exemplos;
- todas as métricas obrigatórias estiverem presentes;
- todas as linhas forem evidência realmente medida.


In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

OUTPUT_COLUMNS = [
    "system",
    "threshold",
    "quality",
    "quality_metric",
    "accuracy",
    "escalation_rate",
    "baseline_share",
    "transformer_share",
    "cost_per_1000",
    "cost_unit",
    "cost_method",
    "latency_ms",
    "latency_p50_ms",
    "latency_p95_ms",
    "source",
    "measured_at",
    "evidence_status",
    "dataset",
    "hardware",
    "sample_size",
    "model_version",
    "notes",
]

if RESOURCES_READY:
    assert len(routing_df) == len(THRESHOLDS)
    assert routing_df["threshold"].tolist() == THRESHOLDS
    assert routing_df["quality"].between(0, 1).all()
    assert routing_df["accuracy"].between(0, 1).all()
    assert routing_df["escalation_rate"].between(0, 1).all()

    measured_at = datetime.now(timezone.utc).isoformat()

    routing_df["source"] = "EDU-ORCH-002"
    routing_df["measured_at"] = measured_at
    routing_df["evidence_status"] = "measured"
    routing_df["dataset"] = "Olist/Polarity; folds 1-8 train, 9 validation, 10 test"
    routing_df["hardware"] = platform.machine()
    routing_df["sample_size"] = len(test_df)
    routing_df["model_version"] = (
        "TF-IDF+MultinomialNB(alpha=1.0) -> "
        "goddiao/distilbert-base-multilingual-cased/pytorch/default/1"
    )
    routing_df["notes"] = (
        "Measured cascade; CPU; Transformer executed only for escalated examples; "
        "cost is computational proxy, not monetary price."
    )

    evidence = routing_df[OUTPUT_COLUMNS].copy()

    out_dir = Path("/kaggle/working/model-evidence")
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / "til-routing-evidence.csv"
    json_path = out_dir / "til-routing-evidence.json"
    manifest_path = out_dir / "manifest.json"
    recovery_path = out_dir / "RECOVERY_COPY.txt"

    evidence.to_csv(csv_path, index=False)
    evidence.to_json(
        json_path,
        orient="records",
        indent=2,
        force_ascii=False,
    )

    def sha256(path):
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    manifest = {
        "experiment": "EDU-ORCH-002",
        "status": "measured",
        "measured_at": measured_at,
        "sample_size": int(len(test_df)),
        "thresholds": [float(x) for x in THRESHOLDS],
        "rows": int(len(evidence)),
        "files": {
            "csv": {
                "path": str(csv_path),
                "sha256": sha256(csv_path),
            },
            "json": {
                "path": str(json_path),
                "sha256": sha256(json_path),
            },
        },
    }

    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    recovery_text = (
        "=== TIL EDU-ORCH-002 RECOVERY COPY ===\n\n"
        + json.dumps(manifest, ensure_ascii=False, indent=2)
        + "\n\n=== CSV ===\n"
        + evidence.to_csv(index=False)
    )
    recovery_path.write_text(recovery_text, encoding="utf-8")

    root_csv = Path("/kaggle/working/til-routing-evidence.csv")
    root_json = Path("/kaggle/working/til-routing-evidence.json")
    root_manifest = Path("/kaggle/working/edu-orch-002-manifest.json")

    shutil.copy2(csv_path, root_csv)
    shutil.copy2(json_path, root_json)
    shutil.copy2(manifest_path, root_manifest)

    bundle_base = Path("/kaggle/working/edu-orch-002-evidence-bundle")
    bundle_zip = Path(
        shutil.make_archive(
            str(bundle_base),
            "zip",
            root_dir=out_dir,
        )
    )

    print("Evidence gate: PASS")
    print("CSV:", csv_path)
    print("JSON:", json_path)
    print("Manifest:", manifest_path)
    print("Recovery copy:", recovery_path)
    print("Bundle:", bundle_zip)
    print("Root CSV:", root_csv)
    print()
    print("=== TIL ROUTING EVIDENCE — RECOVERY COPY ===")
    print(evidence.to_csv(index=False))
else:
    print("Evidence gate: SKIPPED (SMOKE)")


## 7B. Persistência da evidência

A execução oficial deve ser feita com **Save Version → Save & Run All**.

O notebook grava a mesma evidência em formatos redundantes:

```text
/kaggle/working/model-evidence/
├── til-routing-evidence.csv
├── til-routing-evidence.json
├── manifest.json
└── RECOVERY_COPY.txt

/kaggle/working/
├── til-routing-evidence.csv
├── til-routing-evidence.json
├── edu-orch-002-manifest.json
└── edu-orch-002-evidence-bundle.zip
```

Além disso, o CSV completo é impresso no output da célula final. Assim, uma falha de download ou timeout da sessão interativa não exige reexecutar o experimento apenas para recuperar os números.

> A versão salva do Kaggle é a referência oficial de execução. A sessão interativa é apenas ambiente de desenvolvimento.


## 8. Interpretação

Este experimento não escolhe automaticamente o “melhor” threshold.

Ele produz evidência para uma decisão posterior de utility:

```text
measured quality
+ measured escalation
+ measured latency
+ measured runtime
        ↓
utility function
        ↓
engineering decision
```

A Aula 13C poderá então distinguir:

```text
modelos isolados medidos
        ↓
cascades medidos
        ↓
cenários hipotéticos
```

O próximo passo após a coleta é versionar `til-routing-evidence.csv`, revisar a proveniência e integrar a matriz medida à Aula 13C.
